# **The Document Similarity Finder**

- **Objective:** To build a simple but powerful "Document Similarity Finder." This is a core function of vector databases and a common task in data science. You will ingest a small collection of documents, index them, and then find the most similar document to a new, unseen piece of text.
- **Required Libraries:**
•	sentence-transformers (for creating embeddings)
•	faiss-cpu (for ANNS indexing)
•	numpy

- **Scenario:**
You are working with a set of internal company documents. Your task is to build a tool that, when given a new document, can quickly find the most closely related document from your existing knowledge base.


**Student Name:** Supriya Misal

### Part 1: Setup and Data Ingestion

In [ ]:
%pip install -q sentence-transformers faiss-cpu numpy
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

In [ ]:
# ----------------------------------------------------
# Part 1: Setup and Data Ingestion
# ----------------------------------------------------
# 1. Knowledge Base documents
documents = [
    "Quantum computing leverages superposition and entanglement to process information.",
    "Machine learning is a field of AI focused on building systems that learn from data.",
    "The new marketing campaign will focus on social media engagement and influencer partnerships.",
    "Security update: All employees must reset their passwords by the end of the week.",
    "An introduction to quantum mechanics and its principles.",
    "Data-driven decision making is a core tenet of our machine learning strategy.",
]

# 2. Load pre-trained SentenceTransformer model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# 3. Generate embeddings
document_embeddings = embedding_model.encode(documents)

print("Step 1 Complete: All documents have been converted to vector embeddings.")
print(f"Shape of our embedding matrix: {document_embeddings.shape}")

### Part 2: Indexing with FAISS

In [ ]:
# ----------------------------------------------------
# Part 2: Indexing with FAISS
# ----------------------------------------------------
# 1. Dimension of the embeddings (384 for all-MiniLM-L6-v2)
d = document_embeddings.shape[1]

# 2. Create a flat L2 distance index
index = faiss.IndexFlatL2(d)

# 3. Add embeddings as float32 array
index.add(np.array(document_embeddings, dtype=np.float32))

print("\nStep 2 Complete: FAISS index has been built.")
print(f"Number of documents in index: {index.ntotal}")

### Part 3: The Similarity Finder Function

In [ ]:
# ----------------------------------------------------
# Part 3: The Similarity Finder Function
# ----------------------------------------------------
def find_most_similar(query_document: str) -> str:
    """Finds the most similar document in our knowledge base to a new query document."""
    # 1. Generate query embedding
    query_embedding = embedding_model.encode([query_document])

    # 2. Search FAISS index for the single nearest neighbor (k=1)
    distances, indices = index.search(
        np.array(query_embedding, dtype=np.float32), k=1
    )

    # 3. Extract index of the top match
    most_similar_index = indices[0][0]

    # 4. Return the corresponding original document text
    return documents[most_similar_index]


print("\nStep 3 Complete: Similarity finder function is ready.")

### Part 4: Test Your Finder

In [ ]:
# ----------------------------------------------------
# Part 4: Test Your Finder
# ----------------------------------------------------
new_doc = (
    "Our company's new promotional strategy will use TikTok and YouTube."
)
most_similar_doc = find_most_similar(new_doc)

print("\n--- Testing the Similarity Finder ---")
print(f"New Document: '{new_doc}'")
print(f"\nMost Similar Existing Document Found:\n'{most_similar_doc}'")

### Analysis of the test result

The expected closest match is: "The new marketing campaign will focus on social media engagement and influencer partnerships."

The query describes a promotional strategy using TikTok and YouTube, which are social media platforms. Its meaning is therefore similar to the marketing document even though the wording differs. SentenceTransformer captures this semantic similarity in the embeddings.

FAISS `IndexFlatL2` performs exact nearest-neighbor search and returns the document embedding with the smallest squared L2 distance from the query embedding. Compare the actual printed result above with the expected marketing document after running all cells.

### Runtime self-check

Run this cell after Parts 1–4 to verify the embeddings, index, and test-query result.

In [ ]:
assert "document_embeddings" in globals(), "Embeddings are missing; run Part 1 first."
assert isinstance(document_embeddings, np.ndarray) and document_embeddings.ndim == 2, "Embeddings must be a two-dimensional NumPy matrix."
assert document_embeddings.shape[0] == len(documents), "Embedding row count must equal the number of documents."
assert "index" in globals(), "FAISS index is missing; run Part 2 first."
assert index.ntotal == len(documents), "FAISS index count must equal the number of documents."
assert "new_doc" in globals(), "Test query is missing; run Part 4 first."
assert callable(globals().get("find_most_similar")), "Similarity finder is missing; run Part 3 first."
assert isinstance(find_most_similar(new_doc), str), "Similarity finder must return a string for the test query."

print("Runtime self-check passed.")